In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from paths import RAW, PROCESSED, RESULTS, savefig

import geopandas as gpd
import pandas as pd

## 1. Existing Conventional Power Plant Fleet (excluding wind & solar)

Wind and solar are handled separately as weather-driven capacity factor time series
(`04_weather_capacity_factors.ipynb`). Everything else already installed in Austria
(hydro, gas, ...) is treated as an existing, non-extendable generator fleet.

In [ ]:
plants = pd.read_csv(
    RAW / "global-power-plant-database" / "global_power_plant_database.csv",
    low_memory=False,
)

plants_at = plants[plants["country"] == "AUT"].copy()
plants_at = plants_at[~plants_at["primary_fuel"].isin(["Wind", "Solar"])]

plants_at["primary_fuel"].value_counts()

Assign each plant to its model region

Spatial join of plant coordinates against the region polygons produced by
`01_regions_centroids.ipynb`, so region names line up with the rest of the pipeline.

In [ ]:
regions = gpd.read_file(PROCESSED / "gadm_aut_level1.geojson")

plants_gdf = gpd.GeoDataFrame(
    plants_at,
    geometry=gpd.points_from_xy(plants_at.longitude, plants_at.latitude),
    crs="EPSG:4326",
).to_crs(regions.crs)

plants_region = gpd.sjoin(
    plants_gdf,
    regions[["NAME_1", "geometry"]],
    how="left",
    predicate="within",
)

plants_region["NAME_1"].value_counts()

Aggregate to one representative generator per region & technology

Installed capacities are summed per (region, fuel) pair. These existing plants are
marked non-extendable, matching the assignment's simplification.

In [ ]:
existing_generators = (
    plants_region.groupby(["NAME_1", "primary_fuel"], as_index=False)["capacity_mw"]
    .sum()
    .sort_values(["NAME_1", "primary_fuel"])
    .reset_index(drop=True)
)
existing_generators["p_nom_extendable"] = False

existing_generators

Hydro: constant capacity factor (`p_max_pu`) from historical generation / rated capacity

Per the assignment, hydro is modelled as a Generator with `p_max_pu` fixed to the ratio of a
year's historical generation to installed capacity, instead of building a dispatch model for it.
We use 2017 (the most recent year with broad coverage), preferring metered `generation_gwh_2017`
and falling back to the modelled `estimated_generation_gwh_2017` where metered data is missing
(most Austrian hydro plants in the database only have the estimate).

In [ ]:
hydro = plants_region[plants_region["primary_fuel"] == "Hydro"].copy()
hydro["generation_gwh"] = hydro["generation_gwh_2017"].fillna(hydro["estimated_generation_gwh_2017"])

hydro_by_region = hydro.groupby("NAME_1").agg(
    capacity_mw=("capacity_mw", "sum"),
    generation_gwh=("generation_gwh", "sum"),
)
# potential generation at full output: capacity_mw -> GW * 8760 h/year
hydro_by_region["p_max_pu"] = hydro_by_region["generation_gwh"] / (
    hydro_by_region["capacity_mw"] * 8760 / 1000
)

# merge is on NAME_1 alone, so it would otherwise broadcast a region's hydro ratio
# onto every fuel in that region (e.g. gas) - mask it back to hydro rows only
existing_generators = existing_generators.merge(
    hydro_by_region["p_max_pu"], left_on="NAME_1", right_index=True, how="left"
)
existing_generators.loc[existing_generators["primary_fuel"] != "Hydro", "p_max_pu"] = pd.NA

existing_generators.to_csv(PROCESSED / "existing_generators_AUT.csv", index=False)
existing_generators

## 2. Regional Load (population-weighted split of national demand)

The load dataset only has one national time series for Austria (`AT`). We split it across
the 9 model regions in proportion to population as a simple demand-allocation assumption.

In [ ]:
load = pd.read_csv(RAW / "gegis" / "load.csv")

load_at = load[["time", "AT"]].copy()
load_at["time"] = pd.to_datetime(load_at["time"])
load_at = load_at.set_index("time")["AT"]

In [ ]:
# Population figures per region (Statistik Austria), used only to derive load shares.
population = pd.DataFrame({
    "NAME_1": ["Burgenland", "Kärnten", "Niederösterreich", "Oberösterreich", "Salzburg",
               "Steiermark", "Tirol", "Vorarlberg", "Wien"],
    "population": [297583, 564513, 1698796, 1505140, 560710, 1252922, 764102, 401647, 1931593],
})
population["share"] = population["population"] / population["population"].sum()

regional_load = pd.DataFrame(
    {row["NAME_1"]: load_at * row["share"] for _, row in population.iterrows()},
    index=load_at.index,
)

# sanity check: regional shares must sum back to the national total
assert (regional_load.sum(axis=1) - load_at).abs().max() < 1e-6

regional_load.to_csv(PROCESSED / "regional_load_AUT.csv")
regional_load.head()

## 3. Technology Cost Assumptions

Pulls the relevant technology cost assumptions (2030) from the PyPSA `technology-data`
project, for the technologies used in this model.

In [ ]:
costs = pd.read_csv(
    "https://raw.githubusercontent.com/PyPSA/technology-data/master/outputs/costs_2030.csv",
    index_col=[0, 1],
)

# "solar-utility" (not generic "solar") to match the ground-mounted utility-scale
# deployment assumption (3 MW/km2) used in 04_weather_capacity_factors.ipynb and
# the cost table in 06_building_the_PyPSA_model_draft.ipynb
techs = ["hydro", "onwind", "solar-utility", "CCGT", "OCGT"]
costs_table = costs.loc[costs.index.get_level_values(0).isin(techs), "value"].unstack(level=1)

costs_table.to_csv(PROCESSED / "technology_costs.csv")
costs_table